# LDLR cohort for CVFG OR analysis

In Tabet et al. phenotypes examined are:
 - Hyperlipidemia is defined as LDL-C >= 190 mg/dL
 - Familial Hypercholesterolemia (FH)
 - Hypercholesterolemia (HC)
 - Use of cholesterol-lowering drugs
 - CAD (Coronary Artery Disease)
 - Family history of CAD

We will define our control cohort as those without the above

We will define our case cohort as those with
 - Hyperlipidemia
 - FH
 - HC



In [ ]:
from pathlib import Path
import pandas as pd

In [ ]:
# Input file
ldlr_phenotypes_path = Path.home() / 'workspace/scalable-vus-validation' / 'intermediates/LDLR_phenotypes_formatted_censored_ages.csv'
# Output file
ldlr_cohort_path = Path.home() / 'workspace/scalable-vus-validation' / 'gene_cohorts/LDLR cohort.parquet'

In [ ]:
pheno_df = pd.read_csv(ldlr_phenotypes_path, index_col='person_id')
pheno_df

In [ ]:
pheno_df.columns

In [ ]:
for column in ['CAD_code', 'PAD_code', 'FH_code', 'CADFH_code', 'HC_code']:
    display(pheno_df[column].value_counts(dropna=False))

In [ ]:
codes_df = pheno_df[['CAD_code', 'PAD_code', 'FH_code', 'CADFH_code', 'HC_code']]

In [ ]:
# Try to convert these to T/F
binary_df = (~codes_df.isna()).where(codes_df.isna(), codes_df != 0)
binary_df

In [ ]:
binary_df.sum(axis='index')

In [ ]:
binary_df.value_counts().sort_index()

In [ ]:
med_use = pheno_df['any_chol_med'] != False

In [ ]:
hyperlipidemia = (pheno_df['LDL'] >= 90).where(pheno_df['LDL'].isna(), False)

In [ ]:
cohort_df = pd.DataFrame({
    'case': hyperlipidemia | binary_df['FH_code'] | binary_df['HC_code'],
    'control': ~(binary_df.any(axis='columns') | med_use | hyperlipidemia)
})

In [ ]:
cohort_df.value_counts()

In [ ]:
cohort_df[cohort_df['case'] | cohort_df['control']].astype(int).to_parquet(ldlr_cohort_path)